In [28]:
import pandas as pd

df = pd.read_csv('../data/data-students.csv')
df

,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,STU1996,23,Male,2,7.5,20.3,1.89,7.3,10.7,High,8.2,6.3,80.9,No,Average,No,Medium
1996,STU1997,21,Male,2,6.4,16.8,2.14,4.6,5.8,Moderate,4.6,3.5,76.9,Yes,Poor,No,High
1997,STU1998,19,Female,1,4.5,13.1,2.54,7.5,10.0,Low,6.7,8.1,87.9,Yes,NaN,No,High
1998,STU1999,21,Male,3,8.1,22.3,2.62,4.5,6.6,Moderate,6.5,8.4,100.0,No,Good,No,Low


In [29]:
x= df.drop(columns=['Student_ID','Mental_Health_Risk'])
y=df['Mental_Health_Risk']

In [30]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
print("Distribution de y_train :")
print(y_train.value_counts())

print("\nPourcentage :")
print((y_train.value_counts(normalize=True) * 100).round(2))

Distribution de y_train :
Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64

Pourcentage :
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64


In [32]:
num_cols = [
    'Age',
    'Year_of_Study',
    'Sleep_Hours',
    'Study_Hours_Per_Week',
    'GPA',
    'Financial_Stress',
    'Social_Media_Hours',
    'Screen_Time_Hours',
    'Family_Support',
    'Attendance_Rate'
]

cat_cols = [
    'Gender'
]

ord_cols = [
    'Physical_Activity',
    'Diet_Quality'
]

bin_cols = [
    'Part_Time_Job',
    'Previous_Mental_Health_History'
]

In [33]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder ,LabelEncoder , OrdinalEncoder
from sklearn.compose import ColumnTransformer

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[
            ['Low', 'Moderate', 'High'],
            ['Poor', 'Average', 'Good']
        ]
    ))
])

bin_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        drop='if_binary',
        handle_unknown='ignore'
    ))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
    ('ord', ord_pipeline, ord_cols),
    ('bin', bin_pipeline, bin_cols)
])

In [34]:
for c in ord_cols:
    print(f"\n{c}")
    print(x_train[c].unique())
    print(x_train[c].dtype)


Physical_Activity
<ArrowStringArray>
['Moderate', 'High', 'Low']
Length: 3, dtype: str
str

Diet_Quality
<ArrowStringArray>
['Average', 'Poor', 'Good', nan]
Length: 4, dtype: str
str


In [35]:
x_train_scaled = preprocessor.fit_transform(x_train)
x_test_scaled=preprocessor.transform(x_test)


In [36]:
import joblib

joblib.dump(x_train_scaled, '../models/data/x_train_scaled.pkl')
joblib.dump(x_test_scaled, '../models/data/x_test_scaled.pkl')

joblib.dump(y_train, '../models/data/y_train.pkl')
joblib.dump(y_test, '../models/data/y_test.pkl')

['../models/data/y_test.pkl']